# Rivest-Shamir-Adleman Algorithm

RSA(**R**ivest-**S**hamir-**A**dleman) Algorithm is an asymmetric/public-key cryptography algorithm that works using two different keys:
- *Public Key*: It is used for encryption and is known to everyone
- *Private Key*: It is used for decryption and must be kept secret by the receiver

If Person A wants to send a message securely to Person B:
- Person A encrypts the message using Person B's Public Key:
    $$
    Cipher\text{ }Text\text{ }=\text{ }(Message)^e\text{ }\times mod \text{ }n
    $$
    Where $e$ and $n$ are part of the public key
- Person B decrypts the message using their Private Key
    $$
    Message\text{ }=\text{ }(Cipher\text{ }Text)^d\text{ }\times mod \text{ }n
    $$
    Where $d$ and $n$ are part of the public key

$\fbox{IMPORTANT:}\text{ }$ **RSA requires the message to be smaller than n i.e $0\le m < n$**

### Euler's Totient Function $\Phi(n)$

Euler's Totient Function ($\Phi(n)$) for input $n$ returns the number of natural numbers less than $n$ that are coprime to $n$.

$\fbox{Note:}$ **Coprime numbers** are two or more integers that share no common factors other than 1.

## The Algorithm for Key Generation

RSA is based on the fact that it is difficult to factorize a large integer. The Public Key is $(n, e)$ where $n$ and $e$ are publicly known and the Private Key is $(n, d)$.

1. Choose two large prime numbers $p$ and $q$ which should be kept secret.
2. Calculate the product of primes, $n = p * q$.  
    $\fbox{Note:}$ This product is part of the public as well as the private key.
3. Calculate Euler Totient Function:
    $$
    \Phi (n) = \Phi (p) \times \Phi (q)
    $$
    Where, if $x$ is a positive integer and its distinct primes are $p_1, p_2, p_3,...,p_n$ etc. Then:
    $$
    \Phi (x) = x \cdot (1-\frac{1}{p_1}) \cdot (1-\frac{1}{p_2}) \cdot (1-\frac{1}{p_3}) \cdot ... \cdot (1-\frac{1}{p_n})
    $$
4. Choose encryption exponent $e$, such that $1 < e < \Phi(n)$ and $e$ is a coprime with $\Phi(n)$
5. Calculate decryption exponent $d$, such that:
    - $d \times e \equiv\text{ }1\text{ }mod\text{ }\Phi(n)$
    - Even if we get multiple values of $d$ satisfying $d \times e \equiv\text{ }1\text{ }mod\text{ }\Phi(n)$, it does not matter which value we choose as all of them are valid keys and will result into same message on decryption.
6. Lastly, Public Key = $(n, e)$ and Private Key = $(n, d)$.

## Bézout/Linear Diophantine Equations

*Diophantine equations are polynomial equations with integer coefficients where integer solutions are sought.*

Given,

$$ax+by=c,\text{ }\text{ }\text{ }a,b,c\in\Zeta\text{ }(Integers)$$

Let $d = gcd(a,b)$. Solutions to these equations exist only if $d$ divides $c$ or $c\text{ }mod\text{ }d = 0$. If one solution $(x_0,y_0)$ exists, then all solutions are:

$$
x = x_0 + \frac{b}{d}t,\text{ }\text{ }\text{ }y = y_0 - \frac{a}{d}t, \text{ }\text{ }\text{ }t\in\Zeta
$$

$\fbox{Note:}$ The GCD condition is unavoidable since any integer combination $ax+by$ is a multiple of $gcd(a,b)$. Hence, it cannot equal to $c$ UNLESS $c$ is also a multiple of $gcd(a,b)$

### Extended Euclidean Algorithm

Now, we have an equation:

$$ax+by=gcd(a,b)$$

In [ ]:
# Generate two large prime numbers and store them in a .env file
import sympy
import os
from dotenv import set_key
import random

def generate_large_prime(bits=512):
    """Generate a large prime number with the specified bit length."""
    while True:
        num = random.getrandbits(bits)
        if sympy.isprime(num):
            return num

def save_primes_to_env(prime1, prime2, env_file='.env'):
    """Save the generated primes to a .env file."""
    if not os.path.exists(env_file):
        open(env_file, 'w').close()
    set_key(env_file, 'PRIME1', str(prime1))
    set_key(env_file, 'PRIME2', str(prime2))

with open('.env', 'w') as f:
    prime1 = generate_large_prime(8)
    prime2 = generate_large_prime(8)
    save_primes_to_env(prime1, prime2)

In [14]:
def factorise(n):
    """Return the prime factors of the given number n."""
    return sympy.factorint(n)

def EulerTotient(x,y):
    """Calculate Euler's Totient Function for the given prime numbers x, y."""
    return (x - 1) * (y - 1)

def CheckCoprime(a, b):
    """Check if two numbers a and b are coprime."""
    return sympy.gcd(a, b) == 1

In [15]:
# read from .env file 
with open('.env', 'r') as f:
    lines = f.readlines()
    for line in lines:
        if line.startswith('PRIME1='):
            p1 = int(line.split('=')[1].strip().strip("'"))
        elif line.startswith('PRIME2='):
            p2 = int(line.split('=')[1].strip().strip("'"))

Phi_n = EulerTotient(p1,p2)
n = p1 * p2

print(f"Euler's Totient of n: {Phi_n}")
print(f"Product of primes: {n}")

Euler's Totient of n: 39508
Product of primes: 39913


In [17]:
def ChooseCoprime(X):
    """Choose a small coprime integer e for the given Phi_n."""
    e = 3
    try:
        while e < X:
            if CheckCoprime(e, X):
                return e
            e += 2
        raise ValueError("No coprime integer found.")
    except Exception as ex:
        print(f"Error: {ex}")

e = ChooseCoprime(Phi_n)
print(f"Chosen coprime integer e: {e}")

Chosen coprime integer e: 3


In [18]:
def decrypt_private_key(e, Phi_n):
    """Calculate the private key d using the modular inverse of e mod Phi_n."""
    d = sympy.mod_inverse(e, Phi_n)
    return d

d = decrypt_private_key(e, Phi_n)
print(f"Private key d: {d}")

Private key d: 26339


In [19]:
# Store the Public and Private keys in .env file
set_key('.env', 'PUBLIC_KEY', f'({e}, {n})')
set_key('.env', 'PRIVATE_KEY', f'({d}, {n})')

(True, 'PRIVATE_KEY', '(26339, 39913)')

In [20]:
# Plaintext message to number conversion functions
def text_to_numbers(text):
    """Convert plaintext message to a list of numbers based on ASCII values."""
    return [ord(char) for char in text]

def numbers_to_text(numbers):
    """Convert a list of numbers back to plaintext message based on ASCII values."""
    return ''.join(chr(num) for num in numbers)

In [24]:
def encrypt_message(plaintext):
    """Encrypt the plaintext message using the public key (e, n)."""
    with open('.env', 'r') as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith('PUBLIC_KEY='):
                key_str = line.split('=')[1].strip().strip("'")
                e, n = map(int, key_str.strip('()').split(','))
    numbers = text_to_numbers(plaintext)
    # for num in numbers:
    #     cipher = (num ** e) % n
    #     print(cipher)
    cipher = [(num ** e) % n for num in numbers]
    ciphertext = ' '.join(map(str, cipher))

    return ciphertext

def decrypt_message(ciphertext):
    """Decrypt the ciphertext message using the private key (d, n)."""
    with open('.env', 'r') as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith('PRIVATE_KEY='):
                key_str = line.split('=')[1].strip().strip("'")
                d, n = map(int, key_str.strip('()').split(','))
    cipher_numbers = list(map(int, ciphertext.split()))
    decrypted_numbers = [(num ** d) % n for num in cipher_numbers]
    plaintext = numbers_to_text(decrypted_numbers)
    return plaintext

In [25]:
plaintext = str(input("Enter the plaintext message: "))

ciphertext = encrypt_message(plaintext)
print(f"Ciphertext: {ciphertext}")

decrypted_text = decrypt_message(ciphertext)
print(f"Decrypted text: {decrypted_text}")

Ciphertext: 14031 32476 22409 22409 10589 35937 32768 33922 7300 148 4181 32768 148 4181 32768 32499 13005 35147 32768 148 17813 7973 22409 32476 17813 32476 13871 4289 34587 4289 148 10589 13871 17510
Decrypted text: Hello! This is RSA implementation.
